In [1]:
import pandas as pd
import numpy as np
import tools

In [2]:
df_fi = pd.read_parquet("dane/interim/fact_inka_hard_flagged_wazone.parquet")
print(f"Wczytano: {df_fi.shape[0]:,} wierszy, {df_fi['TowId'].nunique():,} TowId")

Wczytano: 4,057,332 wierszy, 12,481 TowId


## Stan magazynowy per TowId

Liczymy stan magazynowy narastająco dla każdego TowId na podstawie chronologii ruchów. Źródło: `fact_inka_hard_flagged_wazone.parquet` (wynik zad_1 — bez martwych/nigdy niesprzedanych towarów — i zad_2 — flaga towarów ważonych).

Reguły ruchu wg `MetodaLiczenia`:
- **IP** — zwykły ruch: `IloscPlus * Mnoznik`
- **IP_IM_DELTA** — różnica: `(IloscPlus - IloscMinus) * Mnoznik`
- **RESET_IP** — BO/remanent: nie jest ruchem, tylko ustawia stan na wartość ze spisu fizycznego (obsłużone osobno)
- **Brak** — nie wpływa na stan (ruch = 0)

In [3]:
# Kontrola: mapowanie TypRuchu / TypDok na regułę liczenia stanu
kontrola = df_fi[
    ['TypRuchu', 'TypDok', 'Dokument', 'WplywNaStan', 'MetodaLiczenia', 'Mnoznik', 'CzyNiechciane']
].drop_duplicates(subset=['TypRuchu', 'TypDok']).sort_values('TypDok')

print(kontrola.to_string(index=False))

    TypRuchu  TypDok       Dokument  WplywNaStan MetodaLiczenia  Mnoznik  CzyNiechciane
   przyjecie       2             PZ         True             IP        1          False
   neutralny       4           ZWFD        False           Brak        0          False
       zwrot       8          ZWPAR         True             IP        1          False
   przyjecie       9             PW         True             IP        1          False
     rozchod      10             RW         True             IP       -1          False
          bo      14             BO         True       RESET_IP        1          False
    remanent      16            REM         True       RESET_IP        1          False
    przecena      18          PRZEC        False           Brak        0           True
    przecena      19     ZAMR_PRZEC        False           Brak        0           True
    sprzedaz      21             DF         True             IP       -1          False
      strata      23            

In [4]:
# ============================================================
# Ruch ilościowy per wiersz wg MetodaLiczenia
# ============================================================
df_fi['RuchIlosciowy'] = 0.0

maska_ip = df_fi['MetodaLiczenia'] == 'IP'
df_fi.loc[maska_ip, 'RuchIlosciowy'] = df_fi.loc[maska_ip, 'IloscPlus'] * df_fi.loc[maska_ip, 'Mnoznik']

maska_delta = df_fi['MetodaLiczenia'] == 'IP_IM_DELTA'
df_fi.loc[maska_delta, 'RuchIlosciowy'] = (
    (df_fi.loc[maska_delta, 'IloscPlus'] - df_fi.loc[maska_delta, 'IloscMinus']) * df_fi.loc[maska_delta, 'Mnoznik']
)

# RESET_IP (bo/remanent) — wartość fizyczna ze spisu, nie ruch.
# Uniwersalna reguła: bierzemy niezerową kolumnę; jeśli obie niezerowe, bierzemy IloscPlus.
maska_reset = df_fi['MetodaLiczenia'] == 'RESET_IP'
df_fi['WartoscFizycznaReset'] = np.where(
    maska_reset,
    np.where(df_fi['IloscPlus'] > 0, df_fi['IloscPlus'], df_fi['IloscMinus']),
    np.nan
)

In [5]:
# ============================================================
# Sortowanie chronologiczne + suma spisu per (TowId, Data)
# Deduplikacja identycznych wartości spisu (te same dokumenty/duplikaty),
# zsumowanie różnych wartości (kilka niezależnych spisów tego samego dnia)
# ============================================================
df_fi = df_fi.sort_values(['TowId', 'Data', 'DokId', 'Kolejnosc']).reset_index(drop=True)
maska_reset = df_fi['MetodaLiczenia'] == 'RESET_IP'

suma_resetu_dzien = (
    df_fi[maska_reset]
    .drop_duplicates(subset=['TowId', 'Data', 'WartoscFizycznaReset'])
    .groupby(['TowId', 'Data'])['WartoscFizycznaReset'].sum()
    .reset_index()
    .rename(columns={'WartoscFizycznaReset': 'SumaResetuDzien'})
)
df_fi = df_fi.merge(suma_resetu_dzien, on=['TowId', 'Data'], how='left')

print(f"Dni ze spisem/BO: {len(suma_resetu_dzien):,}")

Dni ze spisem/BO: 39,022


In [6]:
# ============================================================
# Stan narastająco per TowId, z obsługą resetów (BO/remanent).
# Dla wiersza RESET_IP zapisujemy stan PRZED resetem (do walidacji),
# a bieżący stan ustawiamy na SumaResetuDzien (stan PO spisie fizycznym).
# ============================================================
towid_arr = df_fi['TowId'].to_numpy()
metoda_arr = df_fi['MetodaLiczenia'].to_numpy()
ruch_arr = df_fi['RuchIlosciowy'].to_numpy()
suma_reset_arr = df_fi['SumaResetuDzien'].to_numpy()

n = len(df_fi)
stan_po = np.empty(n)
stan_przed_resetem = np.full(n, np.nan)

stan = 0.0
poprzedni_towid = None
for i in range(n):
    towid = towid_arr[i]
    if towid != poprzedni_towid:
        stan = 0.0
        poprzedni_towid = towid
    if metoda_arr[i] == 'RESET_IP':
        stan_przed_resetem[i] = stan
        stan = suma_reset_arr[i]
    else:
        stan += ruch_arr[i]
    stan_po[i] = stan

df_fi['StanPoTymWierszu'] = stan_po
df_fi['StanPrzedResetem'] = stan_przed_resetem

In [7]:
# ============================================================
# Walidacja: dla prawdziwych remanentów (nie BO) stan PRZED resetem
# powinien zgadzać się z IloscMinus (wartość zdjęta ze stanu księgowego)
# ============================================================
maska_remanent = (df_fi['MetodaLiczenia'] == 'RESET_IP') & (df_fi['TypRuchu'] == 'remanent')
roznica = (df_fi.loc[maska_remanent, 'StanPrzedResetem'] - df_fi.loc[maska_remanent, 'IloscMinus']).round(3)

zgodnosc_pct = (roznica.abs() < 0.01).mean() * 100
print(f"Zgodność remanentów: {zgodnosc_pct:.2f}% z {len(roznica):,} sprawdzonych")

# Flaga jakości — TowId z dużą rozbieżnością przy jakimkolwiek remanencie;
# ich stan magazynowy traktuj z ograniczonym zaufaniem.
duze_rozbieznosci = df_fi.loc[maska_remanent & (roznica.abs() > 10), 'TowId'].unique()
print(f"TowId z rozbieżnością >10 przy remanencie: {len(duze_rozbieznosci)}")

df_fi['StanMniejPewny'] = df_fi['TowId'].isin(duze_rozbieznosci)

Zgodność remanentów: 80.32% z 30,317 sprawdzonych
TowId z rozbieżnością >10 przy remanencie: 854


In [8]:
# ============================================================
# Stan magazynowy per TowId = stan po ostatnim zarejestrowanym ruchu
# ============================================================
ostatni_ruch = df_fi.groupby('TowId').tail(1)[
    ['TowId', 'Data', 'StanPoTymWierszu', 'StanMniejPewny']
].rename(columns={'Data': 'DataOstatniegoRuchu', 'StanPoTymWierszu': 'StanMagazynowy'})

info_towaru = df_fi[['TowId', 'NazwaTow', 'NazwaAsort', 'JestWazony']].drop_duplicates(subset='TowId')

stany_magazynowe_towid = ostatni_ruch.merge(info_towaru, on='TowId', how='left')[
    ['TowId', 'NazwaTow', 'NazwaAsort', 'JestWazony', 'DataOstatniegoRuchu', 'StanMagazynowy', 'StanMniejPewny']
].sort_values('TowId').reset_index(drop=True)

print(f"Stan magazynowy policzony dla {len(stany_magazynowe_towid):,} TowId")
stany_magazynowe_towid.head(10)

Stan magazynowy policzony dla 12,481 TowId


,TowId,NazwaTow,NazwaAsort,JestWazony,DataOstatniegoRuchu,StanMagazynowy,StanMniejPewny
0,15,SOS TABASCO RED 57ML DEVELEY,SOSY GOTOWE /przetwory owocowe,False,2025-12-01,2.0,False
1,49,"WINO B/PW CARLO ROSSI WHITE 10,5% 750ML CEDC","2 WINA, MIÓD NALEWKA DO 18%",False,2026-01-23,8.0,False
2,53,"WINO C/S CARLO ROSSI 9,5% 750ML CEDC","2 WINA, MIÓD NALEWKA DO 18%",False,2026-01-23,5.0,False
3,55,WINO B/S CARLO ROSSI SWEET WHITE MOSCATO 9% 75...,"2 WINA, MIÓD NALEWKA DO 18%",False,2025-11-27,3.0,False
4,56,WINO R/S CARLO ROSSI PINK MOSCATO 9% 750ML CEDC,"2 WINA, MIÓD NALEWKA DO 18%",False,2026-01-13,5.0,False
5,57,WINO C/S MOGEN DAVID POMEGRANATE 10% 750ML BARTEX,"2 WINA, MIÓD NALEWKA DO 18%",False,2026-01-30,5.0,False
6,58,WINO C/PS MOGEN DAVID CONCORD 11% 750ML BARTEX,"2 WINA, MIÓD NALEWKA DO 18%",False,2026-01-30,5.0,False
7,60,WINO B/PS MOGEN DAVID MOSCATO SPOKOJNE 11% 750...,"2 WINA, MIÓD NALEWKA DO 18%",False,2026-01-30,5.0,False
8,62,WINO MOGEN DAVID BLACKBERRY CZ/S 11% 750ML BARTEX,"2 WINA, MIÓD NALEWKA DO 18%",False,2026-01-20,5.0,False
9,64,"RUM CAPITAN MORGAN BLACK LABEL 40% 0,7L",3 ALKOHOLE INNE POWYŻEJ 18%,False,2026-01-31,1.0,False


In [9]:
# Sanity-check przed zapisem
print(stany_magazynowe_towid['StanMagazynowy'].describe())
print(f"\nTowId z ujemnym stanem magazynowym: {(stany_magazynowe_towid['StanMagazynowy'] < 0).sum()}")
print(f"TowId oznaczonych jako mniej pewne (StanMniejPewny): {stany_magazynowe_towid['StanMniejPewny'].sum()}")

count    12481.000000
mean         6.186069
std         23.966820
min       -757.000000
25%          0.000000
50%          0.770000
75%          8.000000
max       1602.000000
Name: StanMagazynowy, dtype: float64

TowId z ujemnym stanem magazynowym: 411
TowId oznaczonych jako mniej pewne (StanMniejPewny): 854


In [10]:
stany_magazynowe_towid.to_parquet(
    "dane/interim/stany_magazynowe_towid.parquet",
    compression='zstd',
    index=False
)
print(f"Zapisano: {stany_magazynowe_towid.shape}")

Zapisano: (12481, 7)


In [11]:
#Suma kontrolna

nazwa_pliku = "stany_magazynowe_towid.parquet"
moj_hash = tools.hash_danych_bezpieczny(f"dane/interim/{nazwa_pliku}", kolumny_sortowania=['TowId'])
print(f"Mój hash (posortowane):   {nazwa_pliku}   {moj_hash}")

Mój hash (posortowane):   stany_magazynowe_towid.parquet   54c1b1de20e69bb0b44655dc6215186ad84bc616df27e4863e33221d6b50f062
